# Answers — sklearn Modelling Essentials

Check after attempting.

In [ ]:
import sys
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.inspection import permutation_importance
ROOT=next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/"reference"/"dataset_loader.py").exists()),Path.cwd())
sys.path.insert(0,str(ROOT/"reference"))
from dataset_loader import load_checked_csv
URL="https://raw.githubusercontent.com/zygmuntz/wine-quality/master/winequality/winequality-red.csv"
df=load_checked_csv("wine_red.csv",URL,["fixed acidity","volatile acidity","alcohol","quality"],1500,{"sep":";"})
X=df.drop(columns="quality"); y=df["quality"]
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=.2,random_state=42)
def metrics(y,p): return {"rmse":mean_squared_error(y,p)**.5,"mae":mean_absolute_error(y,p),"r2":r2_score(y,p)}

## Baseline

In [ ]:
baseline=np.repeat(y_train.mean(),len(y_valid)); print(metrics(y_valid,baseline))

## Ridge

In [ ]:
rows=[]; best=None; best_rmse=np.inf
for a in [.1,1,10]:
    m=Pipeline([("scale",StandardScaler()),("ridge",Ridge(alpha=a))]); m.fit(X_train,y_train); p=m.predict(X_valid); met=metrics(y_valid,p); rows.append({"model":"Ridge","alpha":a,**met})
    if met["rmse"]<best_rmse: best_rmse=met["rmse"]; best=m
display(pd.DataFrame(rows))

## Random Forest

In [ ]:
rf=RandomForestRegressor(n_estimators=300,min_samples_leaf=3,random_state=42,n_jobs=-1); rf.fit(X_train,y_train); pred=rf.predict(X_valid); print(metrics(y_valid,pred))

## Diagnostics

In [ ]:
fig,ax=plt.subplots(figsize=(5,5)); ax.scatter(y_valid,pred,alpha=.35); ax.plot([y_valid.min(),y_valid.max()],[y_valid.min(),y_valid.max()],"--"); ax.set(xlabel="Actual",ylabel="Predicted"); plt.show()
resid=y_valid.to_numpy()-pred; fig,ax=plt.subplots(figsize=(6,4)); ax.scatter(pred,resid,alpha=.35); ax.axhline(0,ls="--"); plt.show()
tmp=pd.DataFrame({"y":y_valid.to_numpy(),"pred":pred}); print(tmp.groupby("y").apply(lambda g: np.sqrt(np.mean((g["y"]-g["pred"])**2))))

## Importance

In [ ]:
pi=permutation_importance(rf,X_valid,y_valid,n_repeats=10,random_state=42,scoring="neg_mean_squared_error"); imp=pd.Series(pi.importances_mean,index=X.columns).sort_values(ascending=False); print(imp.head(5))

## Reasoning
- Baseline tells you whether the model adds value.
- Scaling matters for Ridge because the coefficient penalty depends on units.
- Trees capture thresholds/interactions; Ridge is lower-variance and easier to defend.
- Aggregate RMSE can hide failure on specific target regions.
- Leakage = unavailable/future/test information influencing model fitting or selection.